# Kaggle Notebook Template: Smart Factory Brain Horizon Model Training

This notebook trains the same multi-horizon artifacts expected by the backend:

- `scrap_30m_model.joblib`
- `scrap_30m_scaler.joblib`
- `scrap_30m_feature_list.joblib`
- `scrap_30m_metadata.json`
- `scrap_240m_model.joblib`
- `scrap_240m_scaler.joblib`
- `scrap_240m_feature_list.joblib`
- `scrap_240m_metadata.json`
- `scrap_1440m_model.joblib`
- `scrap_1440m_scaler.joblib`
- `scrap_1440m_feature_list.joblib`
- `scrap_1440m_metadata.json`

It uses `backend_fastapi/train_models_from_csv.py` so exported files stay compatible with the project.

In [ ]:
# Install runtime dependencies in Kaggle
!pip -q install lightgbm scikit-learn joblib shap openpyxl pandas numpy

## Configure Input Paths

Update these paths based on your Kaggle datasets:

- `REPO_INPUT_DIR`: Dataset containing this project files (must include `backend_fastapi/train_models_from_csv.py`)
- `DATA_DIR`: Dataset folder containing machine CSV files (`M*.csv`)
- `MES_WORKBOOK`: Optional MES workbook path used by the trainer
- `PARAMETER_CSV`: Parameter CSV used by `enrich_parameter_csv`

In [ ]:
from pathlib import Path

# Change these for your Kaggle environment
REPO_INPUT_DIR = Path('/kaggle/input/predictive-scrap-ai')
DATA_DIR = Path('/kaggle/input/smart-factory-data')
MES_WORKBOOK = DATA_DIR / 'MES_Manufacturing_M-231_M-356_M-471_M-607_M-612.xlsx'
PARAMETER_CSV = REPO_INPUT_DIR / 'backend_fastapi' / 'AI_cup_parameter_info_cleaned.csv'

# Optional: specify exact files, else trainer auto-discovers M*.csv
MACHINE_FILES = []

OUTPUT_ROOT = Path('/kaggle/working/sfb_export')
HORIZON_DIR = OUTPUT_ROOT / 'horizon'
MODELS_DIR = OUTPUT_ROOT / 'models_misc'
PARAMETER_CSV_V2 = OUTPUT_ROOT / 'AI_cup_parameter_info_cleaned_v2.csv'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HORIZON_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_INPUT_DIR =', REPO_INPUT_DIR)
print('DATA_DIR      =', DATA_DIR)
print('HORIZON_DIR   =', HORIZON_DIR)

In [ ]:
# Validate expected trainer and data layout
trainer_path = REPO_INPUT_DIR / 'backend_fastapi' / 'train_models_from_csv.py'
if not trainer_path.exists():
    raise FileNotFoundError(f'Missing trainer script: {trainer_path}')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Missing DATA_DIR: {DATA_DIR}')

machine_csvs = sorted(DATA_DIR.glob('M*.csv'))
print('Discovered machine csv files:', len(machine_csvs))
for p in machine_csvs[:10]:
    print(' -', p.name)

if len(machine_csvs) == 0 and len(MACHINE_FILES) == 0:
    raise RuntimeError('No machine CSV files found. Provide M*.csv in DATA_DIR or set MACHINE_FILES.')

if not PARAMETER_CSV.exists():
    raise FileNotFoundError(f'Missing parameter csv: {PARAMETER_CSV}')

if not MES_WORKBOOK.exists():
    print('[WARN] MES workbook not found; training can still run with reduced context:', MES_WORKBOOK)

## Run Training

This calls the project trainer directly and writes backend-compatible files to `HORIZON_DIR`.

In [ ]:
import json
import shlex
import subprocess

cmd = [
    'python', str(trainer_path),
    '--data-dir', str(DATA_DIR),
    '--models-dir', str(MODELS_DIR),
    '--horizon-dir', str(HORIZON_DIR),
    '--mes-workbook', str(MES_WORKBOOK),
    '--parameter-csv', str(PARAMETER_CSV),
    '--parameter-csv-v2', str(PARAMETER_CSV_V2),
    '--max-rows-per-machine', '300000',
    '--chunk-size', '50000',
    '--log-level', 'INFO',
]

if MACHINE_FILES:
    cmd.extend(['--machine-files', *MACHINE_FILES])

print('Running command:')
print(' '.join(shlex.quote(x) for x in cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Training failed with exit code {result.returncode}')

# Parse last JSON object from stdout if available
summary = None
for line in reversed(result.stdout.splitlines()):
    line = line.strip()
    if not line:
        continue
    try:
        summary = json.loads(line)
        break
    except Exception:
        pass

if summary is not None:
    print('Parsed training summary:')
    print(json.dumps(summary, indent=2))

In [ ]:
import json

expected_horizons = [30, 240, 1440]
required_suffixes = ['model.joblib', 'scaler.joblib', 'feature_list.joblib', 'metadata.json']

missing = []
for h in expected_horizons:
    for suffix in required_suffixes:
        p = HORIZON_DIR / f'scrap_{h}m_{suffix}'
        if not p.exists():
            missing.append(str(p))

if missing:
    print('[ERROR] Missing expected artifacts:')
    for p in missing:
        print(' -', p)
    raise RuntimeError('Artifact validation failed.')

manifest = {
    'ok': True,
    'horizon_dir': str(HORIZON_DIR),
    'artifacts': sorted(str(p.name) for p in HORIZON_DIR.glob('*')),
}
(OUTPUT_ROOT / 'artifact_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('[OK] Artifact validation passed.')
print(json.dumps(manifest, indent=2))

In [ ]:
# Zip everything for easy Kaggle download
import shutil

zip_base = '/kaggle/working/sfb_horizon_artifacts'
zip_path = shutil.make_archive(base_name=zip_base, format='zip', root_dir=OUTPUT_ROOT)
print('Created zip:', zip_path)
print('Download from Kaggle output files panel.')

## Import Artifacts Into Local Project

After downloading the zip locally:

1. Extract files.
2. Copy `horizon/*` into your project path:
   - `backend_fastapi/models/horizon/`
3. Restart backend.
4. Verify:
   - `GET /api/health`
   - `GET /api/machines/M231-11/chart-data?horizon_minutes=60`
   - `GET /api/fleet/chart-data?horizon_minutes=60`